In [6]:
import numpy as np
import pickle
import pandas as pd
import yaml
import os
from matplotlib import pyplot as plt
import pysindy as ps

def smooth_dynamics(df, order):
    dt = ps.FiniteDifference(order,1,0,drop_endpoints=False)
    ddt = ps.FiniteDifference(order,2,0,drop_endpoints=False)
    _x = df['x'].to_numpy()
    _l = df['l'].to_numpy()
    _t = df['t'].to_numpy()
    new_df = pd.DataFrame(df)
    new_df['xdot'] = dt._differentiate(_x,_t)
    new_df['ldot'] = dt._differentiate(_l,_t)
    new_df['xddot'] = ddt._differentiate(_x,_t)
    return new_df

In [7]:
"""
Load all data from the HSA characterization dataset
"""
characterization = {
    'folder' : r'/home/joseph/workspaces/hsa_hopper/hsa_hopper_control/data/hsa_identification/2024-06-13_1718318116'
}
characterization['data'] = []
characterization['hsa_angle'] = []
keep_columns = ['motor_angle', 'motor_torque', 'times', 'hsa_len', 'dldtheta']
columns = ['x','l','xdot','ldot','xddot','tau','mg','mdot','t','psi','dldx','mode']

# dictionary for remapping column names
column_map = {
    'motor_angle': lambda motor_angle: ('x',motor_angle), 
    'motor_torque': lambda motor_torque: ('tau', motor_torque),
    'times' : lambda times: ('t', times),
    'hsa_len' : lambda hsa_len: ('l', hsa_len),
    'dldtheta' : lambda dldtheta: ('dldx', dldtheta),
}

with open(os.path.join(characterization['folder'], 'experiment_config.yaml'), 'r') as f:
    characterization['experiment_config'] = yaml.load(f, yaml.Loader)

with open(os.path.join(characterization['folder'], 'hardware_config.yaml'), 'r') as f:
    characterization['hardware_config'] = yaml.load(f, yaml.Loader)

order = 2 # finite difference order
decimate = 20 # decimation factor
with open(os.path.join(characterization['folder'], 'data.pickle'), 'rb') as f:
    all_data = pickle.load(f)
    for idx in range(len(all_data['motor_angle'])):
        input_df = pd.DataFrame({key : all_data[key][idx] for key in keep_columns})
        output_df = pd.DataFrame(columns=columns)
        data_remapped = {}
        for key, map in column_map.items():
            new_key, new_data = map(input_df[key].to_numpy())
            output_df[new_key] = new_data[::decimate]
        output_df = smooth_dynamics(output_df, order)
        output_df['psi'] = (np.pi/180)*all_data['hsa_angle'][idx]*np.ones(len(output_df))
        output_df['mg'] = np.zeros(len(output_df))
        output_df['mdot'] = np.zeros(len(output_df))
        output_df['mode'] = 2*np.ones(len(output_df))
        characterization['data'].append(output_df)

In [8]:
from numpy.random import default_rng

ALL_DATA = np.vstack([df.to_numpy() for df in characterization['data']])

rng = default_rng(seed=42)
TRAIN_INDICES = rng.choice(range(ALL_DATA.shape[0]), 8*ALL_DATA.shape[0]//10, replace=False)
DATA_COLUMN_MAP = {columns[i] : i for i in range(len(columns))}
UNIQUE_PSI = np.sort(np.unique(ALL_DATA[:,DATA_COLUMN_MAP['psi']]))

In [9]:
from hsa_hopper.hsa_model import HSAModel
def make_lstsq_data(model, row_indices):
    N = model.num_params()
    M = len(row_indices)
    A = np.zeros((M,2+N))
    b = np.zeros(M)
    for i, idx in enumerate(row_indices):
        l = ALL_DATA[idx,DATA_COLUMN_MAP['l']]
        xdot = ALL_DATA[idx,DATA_COLUMN_MAP['xdot']]
        ldot = ALL_DATA[idx,DATA_COLUMN_MAP['ldot']]
        xddot = ALL_DATA[idx,DATA_COLUMN_MAP['xddot']]
        tau = ALL_DATA[idx,DATA_COLUMN_MAP['tau']]
        dldx = ALL_DATA[idx,DATA_COLUMN_MAP['dldx']]
        psi = ALL_DATA[idx,DATA_COLUMN_MAP['psi']]
        b[i] = tau - ALL_DATA[idx,DATA_COLUMN_MAP['mg']] - ALL_DATA[idx,DATA_COLUMN_MAP['mdot']]
        A[i,0] = xddot
        A[i,1] = xdot
        A[i,2] = dldx*l
        A[i,3] = dldx
        A[i,4] = dldx*ldot
        N_c = model.w_c.shape[0]
        N_d = model.w_d.shape[0]
        z = np.array([l,psi])
        zdot = np.array([ldot,0])
        for j in range(N_c):
            A[i,5+j] = dldx*model.dkc(z,j)[0]
        for j in range(N_d):
            A[i,5+N_c+j] = dldx*model.dkd(z,zdot,j)[0]
    return A,b

In [10]:
"""
Experimental - fitting two potential functions and blending them together with a logistic function
"""

from scipy.optimize import minimize
from scipy.optimize import LinearConstraint
from hsa_hopper.hsa_model import HSAModel

# function to do the model optimization
def governing_equation_lstsq(model):
    N = model.num_params()
    A,b = make_lstsq_data(model,TRAIN_INDICES)

    # rescaling A matrix for better conditioning of constraint evaluation
    SCALE = np.ones(A.shape[1])
    SCALE[0] = 1/1000 # motor inertia rescaling
    SCALE[1] = 1/1000 # motor damping rescaling
    SCALE[2] = 100 # linear spring constant rescaling
    SCALE[3] = 100 # linear force rescaling
    SCALE[4] = 10 # linear damping rescaling

    A_scaled = A*SCALE
    ATA = A_scaled.T@A_scaled
    ATb = A_scaled.T@b
    bTb = np.dot(b,b)
    f = lambda x: x.T@ATA@x+bTb-2*x.T@ATb
    jac = lambda x: 2*ATA@x-2*ATb

    # rescaling matrix for parameters, useful for conditioning
    # the constraint evaluations

    # next constraints - enforce positive weights
    # and positive definiteness of stiffness/damping at basis centers
    C = np.eye(2+N)
    lb = np.array([4.,1,0,-10,0])
    ub = np.array([5.,5,10,10,10])
    lb = np.hstack((lb, -1e3*np.ones(model.w_c.shape[0]), -1e3*np.ones(model.w_d.shape[0])))
    ub = np.hstack((ub, 1e3*np.ones(model.w_c.shape[0]), 1e3*np.ones(model.w_d.shape[0])))
    constraints = [LinearConstraint(C,lb=lb,ub=ub)]

    x0 = np.zeros(2+N)
    x0[0] = 4.5
    result = minimize(f, x0, 
                    jac=jac, 
                    constraints=constraints, 
                    method='SLSQP',
                    options={'maxiter' : 12000}
    )
    result.A = A
    result.b = b
    result.sample_size = b.shape[0]
    result.num_params = model.num_params()
    result.J_x = result.x[0]*SCALE[0]
    result.b_x = result.x[1]*SCALE[1]
    model.K = result.x[2]*SCALE[2]
    model.F0 = result.x[3]*SCALE[3]
    model.b = result.x[4]*SCALE[4]
    N_c = model.w_c.shape[0]
    N_d = model.w_d.shape[0]
    model.w_c = result.x[5:5+N_c]
    model.w_d = result.x[5+N_c:]
    result.model = model
    result.r_sqr = 1-result.fun/np.sum((b-np.average(b))**2)
    result.r_sqr_adj = 1-(1-result.r_sqr)*(result.sample_size-1)/(result.sample_size-result.num_params-1)
    result.mse = result.fun/result.sample_size
    result.rmse = np.sqrt(result.mse)
    result.bic = result.sample_size*np.log(result.mse)+result.num_params*np.log(result.sample_size)
    result.errors = A@(SCALE*result.x)-b
    return result 
    

In [6]:
from hsa_hopper.hsa_model import save_model
y_c = y_d = np.zeros((0,0))
w_c = w_d = np.zeros((y_c.shape[1],))
linear_model = HSAModel(0,0,0,w_c,y_c,w_d,y_d,1,1)
result = governing_equation_lstsq(linear_model)
print(linear_model.attribute_dict())
save_model(linear_model, 'linear_hsa_model.yaml')

{'K': 650.2436762086659, 'F0': -90.05610040010929, 'b': 16.54807365639543, 'w_c': array([], dtype=float64), 'y_c': array([], shape=(0, 0), dtype=float64), 'w_d': array([], dtype=float64), 'y_d': array([], shape=(0, 0), dtype=float64), 'a': 1.0, 's': 1.0}
